# Preparación del dataset para diagnóstico predictivo de fallas informáticas

**Proyecto de apoyo académico — Caso JAVERY / Proyecto de Jimmy Quinga**

Este cuaderno prepara un conjunto de datos público de **Open Repair Alliance (CompCat)** para utilizarlo posteriormente en **Orange Data Mining** y **Power BI**.

## Objetivo del cuaderno
1. Descargar los datos originales de laptops clasificados por tipo de falla.
2. Conservar una copia sin modificar.
3. Auditar calidad, nulos, duplicados y distribución de clases.
4. Limpiar y normalizar variables.
5. Crear variables derivadas útiles.
6. Generar un dataset listo para modelado en Orange.
7. Generar un dataset descriptivo para Power BI.
8. Ejecutar una prueba básica de viabilidad predictiva (opcional).
9. Dejar preparada una sección para integrar los datos reales de JAVERY cuando estén disponibles.

> **Importante:** los datos públicos y los datos de JAVERY deben conservar su procedencia. Nunca deben presentarse los registros externos como si fueran órdenes de trabajo de JAVERY.

**Fuente pública:** Open Repair Alliance — CompCat, repositorio `openrepair/data`  
**Licencia del repositorio:** CC BY-SA 4.0

## 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

print("Librerías cargadas correctamente.")

## 2. Configuración y URLs oficiales

In [ ]:
URL_LAPTOPS = "https://raw.githubusercontent.com/openrepair/data/refs/heads/master/quests/computers/compcat_laptops.csv"
URL_FAULT_TYPES = "https://raw.githubusercontent.com/openrepair/data/refs/heads/master/quests/computers/compcat_fault_types.csv"

BASE_DIR = Path("/content/proyecto_fallas")
RAW_DIR = BASE_DIR / "01_datos_originales"
CLEAN_DIR = BASE_DIR / "02_datos_limpios"
OUTPUT_DIR = BASE_DIR / "03_salidas"

for carpeta in [RAW_DIR, CLEAN_DIR, OUTPUT_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("Carpetas creadas en:", BASE_DIR)

## 3. Descarga y respaldo del dataset original

In [ ]:
df_raw = pd.read_csv(URL_LAPTOPS)
fault_catalog = pd.read_csv(URL_FAULT_TYPES)

raw_path = RAW_DIR / "compcat_laptops_original.csv"
catalog_path = RAW_DIR / "compcat_fault_types_original.csv"

df_raw.to_csv(raw_path, index=False)
fault_catalog.to_csv(catalog_path, index=False)

print("Dataset original:", df_raw.shape)
print("Catálogo de fallas:", fault_catalog.shape)
print("Respaldo guardado en:", raw_path)
display(df_raw.head())

## 4. Catálogo oficial de tipos de falla

In [ ]:
display(fault_catalog)

print("\nTipos de falla presentes en el dataset:")
display(
    df_raw["fault_type"]
    .value_counts(dropna=False)
    .rename_axis("fault_type")
    .reset_index(name="registros")
)

## 5. Auditoría inicial de calidad

In [ ]:
print("DIMENSIONES")
print("Filas:", df_raw.shape[0])
print("Columnas:", df_raw.shape[1])

print("\nCOLUMNAS")
print(list(df_raw.columns))

print("\nTIPOS DE DATOS")
display(df_raw.dtypes.to_frame("tipo"))

print("\nVALORES NULOS")
nulos = (
    df_raw.isna()
    .sum()
    .to_frame("nulos")
    .assign(porcentaje=lambda x: (x["nulos"] / len(df_raw) * 100).round(2))
    .sort_values("porcentaje", ascending=False)
)
display(nulos)

print("\nDUPLICADOS POR ID")
print("IDs duplicados:", df_raw["id_ords"].duplicated().sum())

print("\nREGISTROS TOTALMENTE DUPLICADOS")
print("Duplicados completos:", df_raw.duplicated().sum())

## 6. Copia de trabajo y limpieza básica

In [ ]:
df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

text_cols = [
    "id_ords", "data_provider", "country", "fault_type",
    "product_category", "brand", "repair_status", "problem"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

antes = len(df)
df = df.drop_duplicates().copy()
df = df.drop_duplicates(subset=["id_ords"], keep="first").copy()
despues = len(df)

print("Registros antes:", antes)
print("Registros después de eliminar duplicados:", despues)
print("Eliminados:", antes - despues)

## 7. Limpieza del texto del problema reportado

In [ ]:
def limpiar_texto(texto):
    if pd.isna(texto):
        return pd.NA

    texto = str(texto)
    texto = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", " ", texto)
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)
    texto = re.sub(r"[\r\n\t]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto if texto else pd.NA

df["problem_clean"] = df["problem"].apply(limpiar_texto)
df["problem_chars"] = df["problem_clean"].str.len()
df["problem_words"] = df["problem_clean"].str.split().str.len()

display(df[["problem", "problem_clean", "problem_words"]].head(10))

## 8. Normalización de marca

In [ ]:
df["brand_original"] = df["brand"]

def normalizar_marca(valor):
    if pd.isna(valor):
        return "Unknown"

    marca = str(valor).strip()
    base = marca.lower()

    equivalencias = {
        "hewlett packard": "HP",
        "hewlett-packard": "HP",
        "hp": "HP",
        "lenovo": "Lenovo",
        "ibm lenovo": "Lenovo",
        "dell": "Dell",
        "acer": "Acer",
        "asus": "Asus",
        "apple": "Apple",
        "mac": "Apple",
        "macbook": "Apple",
        "toshiba": "Toshiba",
        "samsung": "Samsung",
        "sony": "Sony",
        "fujitsu": "Fujitsu",
        "medion": "Medion",
        "compaq": "Compaq",
        "unknown": "Unknown"
    }

    if base in equivalencias:
        return equivalencias[base]

    return marca.strip().title()

df["brand"] = df["brand"].apply(normalizar_marca)

print("Marcas más frecuentes tras normalización:")
display(df["brand"].value_counts().head(25).to_frame("registros"))

## 9. Fechas, año de fabricación y antigüedad del equipo

In [ ]:
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")

df["year_of_manufacture"] = pd.to_numeric(
    df["year_of_manufacture"], errors="coerce"
).astype("Int64")

df["event_year"] = df["event_date"].dt.year.astype("Int64")
df["device_age_years"] = df["event_year"] - df["year_of_manufacture"]

mask_age_invalid = (
    (df["device_age_years"] < 0) |
    (df["device_age_years"] > 40)
)
df.loc[mask_age_invalid, "device_age_years"] = pd.NA
df["device_age_years"] = df["device_age_years"].astype("Int64")

display(
    df[["event_date", "year_of_manufacture", "event_year", "device_age_years"]]
    .head(15)
)

## 10. Estandarización de variables categóricas

In [ ]:
df["product_category"] = (
    df["product_category"]
    .fillna("Laptop")
    .replace({"laptop": "Laptop", "LAPTOP": "Laptop"})
)

df["repair_status"] = df["repair_status"].fillna("Unknown").astype(str).str.strip()
df["country"] = df["country"].fillna("Unknown")
df["data_provider"] = df["data_provider"].fillna("Unknown")

# Mantener siempre la procedencia del registro.
df["source_dataset"] = "Open Repair Alliance - CompCat"

display(df.head())

## 11. Diagnóstico de registros útiles para clasificación

In [ ]:
resumen_modelado = pd.DataFrame({
    "criterio": [
        "Registros totales limpios",
        "Con texto de problema",
        "Con tipo de falla",
        "Con texto + tipo de falla",
        "Con texto + falla distinta de Unknown"
    ],
    "registros": [
        len(df),
        df["problem_clean"].notna().sum(),
        df["fault_type"].notna().sum(),
        (df["problem_clean"].notna() & df["fault_type"].notna()).sum(),
        (
            df["problem_clean"].notna()
            & df["fault_type"].notna()
            & ~df["fault_type"].str.lower().eq("unknown")
        ).sum()
    ]
})

display(resumen_modelado)

## 12. Construcción del dataset para modelado

In [ ]:
df_model = df[
    df["problem_clean"].notna()
    & df["fault_type"].notna()
    & ~df["fault_type"].str.lower().eq("unknown")
].copy()

class_counts = df_model["fault_type"].value_counts()

print("Registros para modelado antes de revisar clases pequeñas:", len(df_model))
display(class_counts.to_frame("registros"))

MIN_CLASS_COUNT = 20

clases_validas = class_counts[class_counts >= MIN_CLASS_COUNT].index
clases_excluidas = class_counts[class_counts < MIN_CLASS_COUNT]

print("\nClases con menos de", MIN_CLASS_COUNT, "registros:")
display(clases_excluidas.to_frame("registros"))

df_model_stable = df_model[df_model["fault_type"].isin(clases_validas)].copy()

print("\nDataset estable para modelado:", df_model_stable.shape)
print("Número de clases:", df_model_stable["fault_type"].nunique())

## 13. Análisis exploratorio básico

In [ ]:
conteo_fallas = df_model_stable["fault_type"].value_counts().sort_values()

plt.figure(figsize=(10, 7))
conteo_fallas.plot(kind="barh")
plt.title("Distribución de tipos de falla")
plt.xlabel("Número de registros")
plt.ylabel("Tipo de falla")
plt.tight_layout()
plt.show()

In [ ]:
top_marcas = df_model_stable["brand"].value_counts().head(20).sort_values()

plt.figure(figsize=(10, 7))
top_marcas.plot(kind="barh")
plt.title("20 marcas más frecuentes")
plt.xlabel("Número de registros")
plt.ylabel("Marca")
plt.tight_layout()
plt.show()

In [ ]:
edad = df_model_stable["device_age_years"].dropna().astype(int)

if len(edad) > 0:
    plt.figure(figsize=(9, 5))
    plt.hist(edad, bins=20)
    plt.title("Distribución de antigüedad del equipo al momento de la reparación")
    plt.xlabel("Antigüedad (años)")
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()
else:
    print("No hay suficientes datos de antigüedad para graficar.")

## 14. Revisión de calidad final

In [ ]:
calidad_final = pd.DataFrame({
    "variable": [
        "problem_clean",
        "fault_type",
        "brand",
        "repair_status",
        "year_of_manufacture",
        "device_age_years"
    ],
    "nulos": [
        df_model_stable["problem_clean"].isna().sum(),
        df_model_stable["fault_type"].isna().sum(),
        df_model_stable["brand"].isna().sum(),
        df_model_stable["repair_status"].isna().sum(),
        df_model_stable["year_of_manufacture"].isna().sum(),
        df_model_stable["device_age_years"].isna().sum()
    ]
})

calidad_final["porcentaje_nulos"] = (
    calidad_final["nulos"] / len(df_model_stable) * 100
).round(2)

display(calidad_final)

## 15. Exportación para Orange Data Mining

Se generan dos versiones:

- **`orange_text_model.csv`**: contiene el texto del síntoma/problema y la clase objetivo `fault_type`. Es la versión recomendada si se usa el complemento **Orange3-Text**.
- **`orange_structured_model.csv`**: conserva variables estructuradas y el texto como referencia.

En Orange, **`fault_type` debe configurarse como Target/Class**.

In [ ]:
orange_text = df_model_stable[
    [
        "id_ords",
        "problem_clean",
        "brand",
        "country",
        "repair_status",
        "device_age_years",
        "fault_type",
        "source_dataset"
    ]
].copy()

orange_text = orange_text.rename(columns={"problem_clean": "problem"})

orange_structured = df_model_stable[
    [
        "id_ords",
        "product_category",
        "brand",
        "country",
        "repair_status",
        "year_of_manufacture",
        "device_age_years",
        "problem_clean",
        "fault_type",
        "source_dataset"
    ]
].copy()

orange_text_path = OUTPUT_DIR / "orange_text_model.csv"
orange_structured_path = OUTPUT_DIR / "orange_structured_model.csv"

orange_text.to_csv(orange_text_path, index=False, encoding="utf-8-sig")
orange_structured.to_csv(orange_structured_path, index=False, encoding="utf-8-sig")

print("Archivo Orange Text:", orange_text_path)
print("Archivo Orange estructurado:", orange_structured_path)
print("Registros exportados:", len(orange_text))

## 16. Exportación para Power BI

In [ ]:
powerbi_cols = [
    "id_ords",
    "data_provider",
    "country",
    "product_category",
    "brand",
    "brand_original",
    "repair_status",
    "year_of_manufacture",
    "event_date",
    "event_year",
    "device_age_years",
    "problem_clean",
    "fault_type",
    "source_dataset"
]

df_powerbi = df[powerbi_cols].copy()

powerbi_path = OUTPUT_DIR / "powerbi_reparaciones_laptops.csv"
df_powerbi.to_csv(powerbi_path, index=False, encoding="utf-8-sig")

print("Archivo Power BI:", powerbi_path)
print("Registros:", len(df_powerbi))

## 17. Prueba rápida de viabilidad predictiva — opcional

Esta sección **no sustituye Orange Data Mining**. Solo sirve para comprobar que el texto del problema contiene señal predictiva respecto al tipo de falla.

Se usa:
- TF-IDF.
- Regresión logística.
- Validación cruzada estratificada de 5 particiones.
- Accuracy, Precision macro, Recall macro y F1 macro.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

X = df_model_stable["problem_clean"].astype(str)
y = df_model_stable["fault_type"].astype(str)

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            max_features=15000
        )
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    pipeline,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1_macro": "f1_macro",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro"
    },
    n_jobs=-1
)

resultado_cv = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision macro", "Recall macro", "F1 macro"],
    "Promedio": [
        scores["test_accuracy"].mean(),
        scores["test_precision_macro"].mean(),
        scores["test_recall_macro"].mean(),
        scores["test_f1_macro"].mean()
    ],
    "Desviación estándar": [
        scores["test_accuracy"].std(),
        scores["test_precision_macro"].std(),
        scores["test_recall_macro"].std(),
        scores["test_f1_macro"].std()
    ]
})

display(resultado_cv.round(4))

## 18. Preparación para incorporar la base real de JAVERY

Cuando Jimmy entregue el Excel original, **no se debe pegar directamente sobre CompCat**.

Primero se realizará:

1. Respaldo del archivo original de JAVERY.
2. Auditoría de columnas y calidad.
3. Anonimización de cualquier dato personal.
4. Homologación de variables.
5. Creación de `source_dataset = "JAVERY"`.
6. Comparación de categorías de diagnóstico con `fault_type`.
7. Integración únicamente de variables compatibles.
8. Validación separada con casos de JAVERY.

La estrategia recomendada será comparar, si el número de registros lo permite:

- **Modelo A:** entrenado únicamente con datos de JAVERY.
- **Modelo B:** entrenado con datos externos compatibles.
- **Modelo C:** entrenamiento enriquecido y evaluación final separada sobre JAVERY.

## 19. Plantilla de carga futura del Excel de JAVERY

In [ ]:
# EJECUTAR SOLO CUANDO SE DISPONGA DEL ARCHIVO REAL DE JAVERY.

# from google.colab import files
# uploaded = files.upload()
# nombre_archivo = next(iter(uploaded))
# df_javery_raw = pd.read_excel(nombre_archivo)
#
# print("Dimensiones JAVERY:", df_javery_raw.shape)
# display(df_javery_raw.head())
# display(df_javery_raw.isna().sum().to_frame("nulos"))
#
# NO integrar todavía.
# Primero se revisarán los nombres reales de columnas y se construirá
# un diccionario de homologación validado.

## 20. Descargar los archivos generados

In [ ]:
from google.colab import files

print("Archivos disponibles:")
print("1.", orange_text_path)
print("2.", orange_structured_path)
print("3.", powerbi_path)

# Descomenta según lo que quieras descargar:
# files.download(str(orange_text_path))
# files.download(str(orange_structured_path))
# files.download(str(powerbi_path))

## Observaciones metodológicas importantes

- La base pública de CompCat **no pertenece a JAVERY**.
- Su uso debe declararse expresamente en el documento de titulación.
- El dataset público es útil como conjunto complementario y como evidencia de viabilidad.
- La validación principal del caso de estudio debe hacerse, en la medida de lo posible, con registros reales de JAVERY.
- No deben crearse registros ficticios para aumentar artificialmente la muestra.
- No se recomienda usar `Accuracy` como única métrica.
- Antes de la defensa debe quedar documentado qué registros se usaron para entrenamiento, validación y prueba.